# Assignment 7 — NLP-I (UCS420: Cognitive Computing)

**Name:** Jasmandeep Sharma  
**Roll Number:** 1024170269

> Important: this assignment requires the exact ticket dataset generated by the instructor's `UCS420_LA9_Starter.ipynb`. Paste/run its original setup and function cells unchanged in the setup cell below. Do not modify `generate_dataset()` or `process_ticket()`. The analysis here uses those generated tickets rather than inventing replacement data.

## Setup — use the official starter notebook

In [ ]:
# Paste the original starter notebook's imports, TEST_LINES, generate_dataset(), process_ticket(), and dataset-generation cells here unchanged.
# Run the original generation step using roll number 1024170269.
# Make sure the generated ticket collection is available as `tickets`.
# If the starter notebook uses a different variable name, assign that collection to `tickets` without changing the generated data.

# Required objects after running the starter code:
#   tickets          -> generated raw IT-support tickets (strings or records containing text)
#   TEST_LINES       -> provided test lines
#   process_ticket() -> original preprocessing function
#   generate_dataset() -> original dataset generator

## Imports and helpers

In [ ]:
import re
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from collections import Counter
from nltk.tokenize import RegexpTokenizer, word_tokenize
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer

for resource in ['punkt', 'punkt_tab', 'stopwords', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'wordnet', 'omw-1.4']:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

def get_ticket_text(ticket):
    """Get raw text from common ticket representations without changing the data."""
    if isinstance(ticket, str):
        return ticket
    if isinstance(ticket, dict):
        for key in ('text', 'ticket', 'description', 'raw_text', 'message', 'body'):
            if key in ticket and isinstance(ticket[key], str):
                return ticket[key]
    return str(ticket)

def basic_tokens(text):
    return re.findall(r'\w+|[^\w\s]', text, flags=re.UNICODE)

def sentence_count(text):
    chunks = re.split(r'(?<=[.!?])\s+', text.strip())
    return sum(bool(chunk.strip()) for chunk in chunks)

def get_final_tokens(processed):
    """Extract final tokens from common dictionary/list outputs of process_ticket()."""
    if isinstance(processed, dict):
        for key in ('final_tokens', 'final_token', 'clean_tokens', 'tokens_final'):
            if key in processed:
                return list(processed[key])
    if isinstance(processed, (list, tuple)):
        # If the function returns a list/tuple of tokens, use it directly.
        if all(isinstance(x, str) for x in processed):
            return list(processed)
    raise ValueError('Could not identify final tokens in process_ticket() output. Inspect the output and update only this adapter, not the starter function.')

## Q1. Preprocessing — process every ticket and build the table

In [ ]:
# Confirm that the official starter notebook has been run.
required_names = ['tickets', 'TEST_LINES', 'process_ticket']
missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError('Run/paste the original starter notebook cells first. Missing: ' + ', '.join(missing))

ticket_texts = [get_ticket_text(ticket) for ticket in tickets]
processed_tickets = [process_ticket(ticket) for ticket in tickets]
print('Number of tickets:', len(ticket_texts))
print('Example process_ticket() output:')
display(processed_tickets[0])

In [ ]:
preprocessing_rows = []
for i, (text, processed) in enumerate(zip(ticket_texts, processed_tickets)):
    tokens = basic_tokens(text)
    punctuation_tokens = [token for token in tokens if re.fullmatch(r'[^\w\s]+', token)]
    words = [token for token in tokens if re.search(r'\w', token)]
    final_tokens = get_final_tokens(processed)
    preprocessing_rows.append({
        'ticket_id': i + 1,
        'number_of_sentences': sentence_count(text),
        'number_of_tokens': len(tokens),
        'number_of_punctuation_tokens': len(punctuation_tokens),
        'number_of_stopwords': sum(1 for token in words if token.lower() in STOPWORDS),
        'number_of_unique_tokens': len(set(token.lower() for token in tokens)),
        'number_of_final_tokens': len(final_tokens)
    })
preprocessing_df = pd.DataFrame(preprocessing_rows)
count_columns = [col for col in preprocessing_df.columns if col != 'ticket_id']
total_row = {'ticket_id': 'TOTAL'}
for col in count_columns:
    total_row[col] = preprocessing_df[col].sum()
preprocessing_df = pd.concat([preprocessing_df, pd.DataFrame([total_row])], ignore_index=True)
display(preprocessing_df)

In [ ]:
# Display tokens for one selected ticket (change selected_ticket_index if desired).
selected_ticket_index = 0
selected_text = ticket_texts[selected_ticket_index]
selected_tokens = basic_tokens(selected_text)
selected_punctuation = [t for t in selected_tokens if re.fullmatch(r'[^\w\s]+', t)]
selected_stopwords = [t for t in selected_tokens if re.search(r'\w', t) and t.lower() in STOPWORDS]
selected_final_tokens = get_final_tokens(processed_tickets[selected_ticket_index])
print('Raw ticket:', selected_text)
print('Actual tokens:', selected_tokens)
print('Punctuation tokens:', selected_punctuation)
print('Stopwords:', selected_stopwords)
print('Final tokens from process_ticket():', selected_final_tokens)

## Q2. Regex extraction and replacement

In [ ]:
all_raw_text = '\n'.join(ticket_texts)

# Extract from raw tickets
long_words = re.findall(r'\b[A-Za-z]{6,}\b', all_raw_text)
numbers = re.findall(r'\b\d+(?:\.\d+)+|\b\d+\b', all_raw_text)
capitalized_words = re.findall(r'\b[A-Z][a-zA-Z]*\b', all_raw_text)
vowel_words = re.findall(r'\b[AEIOUaeiou][A-Za-z]*\b', all_raw_text)
print('Words longer than 5 letters:', long_words)
print('\nNumbers (including decimals/versions):', numbers)
print('\nCapitalized words:', capitalized_words)
print('\nWords beginning with a vowel:', vowel_words)

In [ ]:
# Replace email addresses, URLs, and phone numbers.
# The same patterns are applied to the provided TEST_LINES below.
EMAIL_PATTERN = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b'
URL_PATTERN = r'\b(?:https?://|www\.)\S+'
PHONE_PATTERN = r'(?<!\w)(?:\+?\d{1,3}[ .-]?)?(?:\(?\d{3,5}\)?[ .-]?)\d{3,5}[ .-]?\d{3,5}(?!\w)'

def redact_sensitive(text):
    text = re.sub(EMAIL_PATTERN, '<EMAIL>', text)
    text = re.sub(URL_PATTERN, '<URL>', text)
    text = re.sub(PHONE_PATTERN, '<PHONE>', text)
    return text

print('Redacted tickets:')
for line in ticket_texts:
    print(redact_sensitive(line))

print('\nRegex replacement on TEST_LINES:')
for line in TEST_LINES:
    print('Original:', line)
    print('Redacted:', redact_sensitive(line))

## Q3. Custom tokenization and comparison with word_tokenize()

In [ ]:
# One RegexpTokenizer pattern: URLs, emails, phone numbers, versions, decimals,
# contractions, hyphenated words, words, then standalone punctuation.
TOKEN_PATTERN = r"https?://[^\s]+|www\.[^\s]+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|(?:\+?\d[\d(). -]{6,}\d)|\d+(?:\.\d+)+|\d+|[A-Za-z]+(?:[-'][A-Za-z]+)*|[^\w\s]"
custom_tokenizer = RegexpTokenizer(TOKEN_PATTERN)

def my_tokenize(text):
    return custom_tokenizer.tokenize(text)

print('TEST_LINES custom-tokenizer comparison:')
for line in TEST_LINES:
    print('\nText:', line)
    print('my_tokenize:', my_tokenize(line))
    print('word_tokenize:', word_tokenize(line))

print('\nGenerated ticket custom-tokenizer comparison:')
for i, text in enumerate(ticket_texts, start=1):
    print(f'Ticket {i}:', my_tokenize(text))

**At least three likely differences (confirm them against your actual TEST_LINES):**

1. The custom tokenizer keeps contractions such as `isn't` together; `word_tokenize()` may split them.
2. The custom tokenizer preserves hyphenated terms such as `Wi-Fi` and `state-of-the-art` as one token.
3. The custom tokenizer is designed to keep decimals and version strings such as `95.5` and `2.0.1` intact.
4. It also prioritizes complete email addresses, URLs and phone numbers as single tokens. Exact outputs depend on the supplied test lines and NLTK version.

## Q4. Stemming and POS-aware lemmatization

In [ ]:
final_tokens_all = []
for processed in processed_tickets:
    final_tokens_all.extend(get_final_tokens(processed))
unique_final_tokens = sorted(set(str(token).lower() for token in final_tokens_all if str(token).strip()))

porter = PorterStemmer()
lancaster = LancasterStemmer()
# Choose a suffix supported by common inflected forms; inspect dataset words below to justify it.
regexp_stemmer = RegexpStemmer(r'(ing|ed|es|s)$', min=4)
lemmatizer = WordNetLemmatizer()

def penn_to_wordnet(tag):
    if tag.startswith('J'):
        return 'a'
    if tag.startswith('V'):
        return 'v'
    if tag.startswith('N'):
        return 'n'
    if tag.startswith('R'):
        return 'r'
    return 'n'

tagged = nltk.pos_tag(unique_final_tokens)
pos_map = {word: penn_to_wordnet(tag) for word, tag in tagged}
stem_rows = []
for word in unique_final_tokens:
    porter_out = porter.stem(word)
    lancaster_out = lancaster.stem(word)
    regexp_out = regexp_stemmer.stem(word)
    lemma_out = lemmatizer.lemmatize(word, pos=pos_map.get(word, 'n'))
    outputs = [porter_out, lancaster_out, regexp_out, lemma_out]
    if len(set(outputs)) > 1:
        stem_rows.append({'word': word, 'Porter': porter_out, 'Lancaster': lancaster_out,
                          'Regexp': regexp_out, 'WordNet_lemma_POS': lemma_out})
differences_df = pd.DataFrame(stem_rows)
print('Only words whose outputs differ:')
display(differences_df)

print('Dataset words ending in candidate suffixes (for RegexpStemmer justification):')
display(pd.DataFrame({'word': unique_final_tokens})[
    pd.DataFrame({'word': unique_final_tokens})['word'].str.contains(r'(ing|ed|es|s)$', regex=True)
].head(30))

**Regexp Stemmer suffix justification:** The pattern removes common endings `ing`, `ed`, `es`, or `s` from tokens long enough to avoid stripping very short words. Use the printed dataset examples to mention at least two actual words from your generated tickets and explain why this suffix pattern fits them.

## Q5. Corpus statistics, top words, Zipf-style plot and bigrams

In [ ]:
s1 = [token.lower() for text in ticket_texts for token in my_tokenize(text)]
s2 = [token for token in s1 if not re.fullmatch(r'[^\w\s]+', token)]
s3 = [str(token).lower() for token in final_tokens_all]
s4 = [porter.stem(token) for token in s3]

def corpus_stats(tokens, name):
    total = len(tokens)
    unique = len(set(tokens))
    ttr = unique / total if total else 0
    return {'Corpus': name, 'Total tokens': total, 'Unique tokens': unique, 'TTR': ttr}

stats_df = pd.DataFrame([
    corpus_stats(s1, 'S1: all tokens'),
    corpus_stats(s2, 'S2: without punctuation'),
    corpus_stats(s3, 'S3: final tokens'),
    corpus_stats(s4, 'S4: S3 after Porter stemming')
])
display(stats_df)

print('Top 10 words in S1:')
display(pd.DataFrame(Counter(s1).most_common(10), columns=['Word', 'Frequency']))
print('Top 10 words in S3:')
display(pd.DataFrame(Counter(s3).most_common(10), columns=['Word', 'Frequency']))

# Plot log(rank) vs log(frequency) for S3
freqs = sorted(Counter(s3).values(), reverse=True)
ranks = np.arange(1, len(freqs) + 1)
nonzero = np.array(freqs) > 0
plt.figure(figsize=(7, 5))
plt.plot(np.log(ranks[nonzero]), np.log(np.array(freqs)[nonzero]), marker='o', linestyle='none')
plt.xlabel('log(rank)')
plt.ylabel('log(frequency)')
plt.title('Log(rank) vs Log(frequency) for S3')
plt.grid(True)
plt.show()

def top_bigrams(tokens, n=5):
    return Counter(zip(tokens, tokens[1:])).most_common(n)

print('Top 5 bigrams in S1:')
display(pd.DataFrame([(f'{a} {b}', count) for (a, b), count in top_bigrams(s1)], columns=['Bigram', 'Frequency']))
print('Top 5 bigrams in S3:')
display(pd.DataFrame([(f'{a} {b}', count) for (a, b), count in top_bigrams(s3)], columns=['Bigram', 'Frequency']))

## Q6. Interpretation

Answer these briefly after running the notebook and checking your dataset-specific results:

**(a) Why does TTR change from S1 to S4?**  
TTR is the ratio of unique tokens to total tokens. Removing punctuation changes the token pool, and stemming merges related word forms (for example, inflections may share one stem), which can reduce the number of unique types. The exact direction and size depend on this generated dataset.

**(b) One difference between `word_tokenize()` and `my_tokenize()`:**  
Use an actual example printed in Q3. The custom tokenizer is designed to preserve contractions, hyphenated words, decimals, versions, emails, URLs and phone numbers as single tokens.

**(c) One undesirable stemming result:**  
Choose a real example from the Q4 comparison table where a stemmer produces an awkward or non-word form that loses some readability or meaning.

**(d) One case where POS-aware lemmatization preserves meaning better:**  
Choose a verb or noun from the Q4 table where WordNet lemmatization returns a valid dictionary base form while a stemmer returns only a shortened stem. Use the actual outputs from your generated tickets.